# Notebook 04 — Multi-Site Analysis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polar-bear-after-lunch/AareML/blob/main/notebooks/04_multisite_analysis.ipynb)

Evaluates zero-shot transfer and per-gauge retraining of the LSTM across all 16 Swiss gauges with sufficient DO coverage. Requires checkpoint from notebook 03.

# AareML — 04 Multi-Site Analysis

**Project:** AareML — Predicting River Water Quality in Swiss Catchments  

---

## Overview

This notebook is the **core novel contribution** of AareML.
We evaluate two generalisation strategies across all gauges with meaningful
dissolved oxygen data:

| Strategy | Description |
|----------|-------------|
| **Zero-shot transfer** | Load the single-site model trained on gauge 2473 and apply it directly to all other gauges — tests how well the model generalises without retraining. |
| **Leave-one-out (LOO) per-gauge** | Retrain a fresh model on every gauge independently — gives an upper bound on per-gauge performance. |

**Then:** correlate per-gauge RMSE with catchment attributes
(area, elevation, land cover, etc.) to understand what makes some sites
harder to predict than others.


## 0 · Setup


In [1]:
# ── Colab setup (auto-runs only in Google Colab) ──────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import os
    from pathlib import Path

    # ── 1. Clone repo ──────────────────────────────────────────────────────
    if not Path('AareML').exists():
        os.system('git clone https://github.com/polar-bear-after-lunch/AareML.git')
    if not str(Path.cwd()).endswith('AareML'):
        os.chdir('AareML')

    # ── 2. Install dependencies ────────────────────────────────────────────
    os.system('pip install -q -r requirements.txt')

    # ── 3. Mount Google Drive for persistent data storage ─────────────────
    from google.colab import drive
    drive.mount('/content/drive')

    # Data stored in Drive so it survives across sessions (~360 MB total)
    DRIVE_DATA = Path('/content/drive/MyDrive/AareML_data')
    LOCAL_DATA = Path('data')
    LOCAL_DATA.mkdir(exist_ok=True)

    # ── 4. CAMELS-CH-Chem ─────────────────────────────────────────────────
    DRIVE_CAMELS = DRIVE_DATA / 'camels-ch-chem'
    LOCAL_CAMELS = LOCAL_DATA / 'camels-ch-chem'

    if DRIVE_CAMELS.exists():
        # Already in Drive — symlink to local path (fast, no re-download)
        if not LOCAL_CAMELS.exists():
            os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem loaded from Google Drive.')
    else:
        # First time — download to Drive
        print('Downloading CAMELS-CH-Chem to Google Drive (~165 MB, one-time)...')
        DRIVE_DATA.mkdir(parents=True, exist_ok=True)
        os.system(
            'wget -q --show-progress -O /tmp/camels.zip '
            '"https://zenodo.org/api/records/14980027/files/camels-ch-chem.zip/content"'
        )
        os.system(f'unzip -q /tmp/camels.zip -d {DRIVE_CAMELS}')
        os.system('rm /tmp/camels.zip')
        os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem saved to Google Drive for future sessions.')

    print(f'Setup complete. Working directory: {os.getcwd()}')


In [2]:
# ── CPU thread optimisation ────────────────────────────────────────────────
# PyTorch LSTM on CPU is fastest with 4-6 threads — beyond that,
# thread synchronisation overhead outweighs the parallelism gains.
import os, multiprocessing
N_CPU = multiprocessing.cpu_count()
N_THREADS = min(N_CPU, 6)  # clamp to 6 — empirically optimal on macOS
os.environ['OMP_NUM_THREADS']      = str(N_THREADS)
os.environ['MKL_NUM_THREADS']      = str(N_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_THREADS)
print(f'CPU cores: {N_CPU} logical | Using {N_THREADS} threads for PyTorch')


CPU cores: 6 logical | Using 6 threads for PyTorch


In [3]:
import sys, time
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import DataLoader

from src.config  import *
from src.data    import (
    load_gauge, load_metadata, preprocess,
    train_val_test_split, make_windows, gauges_with_do,
)
from src.metrics import mean_rmse, mean_mae, nse, kge, rmse_per_step
from src.model   import (
    RiverDataset, Seq2SeqLSTM,
    train_model, predict, get_y_true, load_checkpoint,
    EASeq2SeqLSTM, EARiverDataset, train_ea_model,
)
from src.impute  import SATSImputer

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cpu


In [4]:
# Metadata normalisation is handled inline where needed (cell 18)
pass


In [5]:
# ── GPU / DataLoader configuration ────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# LOCAL_TEST: True = fast CPU verification, False = full run
LOCAL_TEST = DEVICE.type == 'cpu'
if LOCAL_TEST:
    print('LOCAL_TEST mode: reduced epochs/trials for quick verification')


Device: cpu
LOCAL_TEST mode: reduced epochs/trials for quick verification


## 1 · Select Multi-Site Gauges

Only gauges with ≥ 10% DO coverage and a minimum number of usable
test windows are included.


In [6]:
# Gauges with at least 10% DO data
do_gauges = gauges_with_do(min_coverage=0.10)
print(f'Gauges with ≥10% DO coverage: {len(do_gauges)}')

# Pre-screen: require at least 50 valid test windows
MIN_TEST_WINDOWS = 50
valid_gauges = []

for gid in do_gauges:
    try:
        raw  = load_gauge(gid)
        data = preprocess(raw)
        _, _, test = train_val_test_split(data)
        train_m = (
    pd.concat([data.loc[:TRAIN_END][FEATURES].mean(),
               data.loc[:TRAIN_END][TARGETS].mean()])
    .groupby(level=0).first()
)
        _, _, d_test = make_windows(test, train_m)
        if len(d_test) >= MIN_TEST_WINDOWS:
            valid_gauges.append(gid)
    except Exception as e:
        print(f'  Skip {gid}: {e}')

print(f'Valid gauges (≥{MIN_TEST_WINDOWS} test windows): {len(valid_gauges)}')
print('Gauge IDs:', valid_gauges)


[data] load_gauge 2009: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2011: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2016: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2018: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2019: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2029: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2030: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2033: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2034: 

[data] load_gauge 2091: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2104: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']


[data] load_gauge 2106: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2109: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']


[data] load_gauge 2112: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2113: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2126: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2130: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2135: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2139: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2143: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2150: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2152: 

[data] load_gauge 2205: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] load_gauge 2210: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']


...

[data] make_windows: 1203 windows, X=(1203, 21, 4), y=(1203, 14, 2), date range 2017-01-22 → 2020-12-18


[data] load_gauge 2243: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.619, 'ec_sensor': 0.615, 'O2C_sensor': 0.613}
[data] split: train=12418, val=731, test=1461
  Skip 2243: make_windows produced 0 valid windows. Check NaN coverage — the target columns may be entirely missing or the DataFrame is shorter than lookback + horizon = 35 days.
[data] load_gauge 2410: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.263, 'pH_sensor': 0.268, 'ec_sensor': 0.263, 'O2C_sensor': 0.265}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1392 windows, X=(1392, 21, 4), y=(1392, 14, 2), date range 2017-01-22 → 2020-12-18
[data] load_gauge 2415: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14

[data] make_windows: 1348 windows, X=(1348, 21, 4), y=(1348, 14, 2), date range 2017-01-22 → 2020-12-18


[data] load_gauge 2613: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.35, 'pH_sensor': 0.023, 'ec_sensor': 0.008, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461


[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18
Valid gauges (≥50 test windows): 12
Gauge IDs: ['2009', '2016', '2018', '2044', '2085', '2143', '2174', '2410', '2415', '2462', '2473', '2613']


## 2 · Load Single-Site Checkpoint


In [7]:
ckpt_path = RESULTS_DIR / 'lstm_single_site_best.pt'

if not ckpt_path.exists():
    print('WARNING: checkpoint not found — run notebook 03 first.')
    print('Falling back to default model (hidden=64, layers=2, dropout=0.2)')
    transfer_params = dict(hidden=64, n_layers=2, dropout=0.2)
    transfer_ckpt   = None
else:
    transfer_ckpt = load_checkpoint(ckpt_path, device=DEVICE)
    transfer_params = transfer_ckpt['best_params']
    print(f'Loaded checkpoint: {transfer_params}')


[model] load_checkpoint: loaded from /Users/amber/VS Code/polar-bear-after-lunch/AareML/notebooks/../results/lstm_single_site_best.pt, keys=['model_state', 'best_params', 'feat_scaler_mean', 'feat_scaler_scale', 'tgt_scaler_mean', 'tgt_scaler_scale']
Loaded checkpoint: {'hidden': 256, 'n_layers': 2, 'dropout': 0.1465447373174767, 'lr': 0.0005954553793888993, 'batch_size': 64}


In [8]:
# ── Constants derived from config (used throughout notebook) ──────────────
N_FEAT = len(FEATURES)   # number of input features (4)
N_TGT  = len(TARGETS)    # number of output targets (2)

# ── Focus gauge scalers + val windows (needed for EA-LSTM early stopping) ─
raw_focus   = load_gauge(FOCUS_GAUGE)
data_focus  = preprocess(raw_focus)
train_focus, val_focus, _ = train_val_test_split(data_focus)
train_means_focus = (
    pd.concat([train_focus[FEATURES].mean(), train_focus[TARGETS].mean()])
    .groupby(level=0).first()
)

feat_sc_focus = StandardScaler().fit(
    train_focus[FEATURES].fillna(train_means_focus[FEATURES])
)
tgt_sc_focus  = StandardScaler().fit(
    train_focus[TARGETS].fillna(train_means_focus[TARGETS])
)

X_val_raw, y_val_raw, _ = make_windows(val_focus, train_means_focus)
Nv, Lv, _ = X_val_raw.shape
_, Hv, _  = y_val_raw.shape
X_val_sc = feat_sc_focus.transform(
    X_val_raw.reshape(-1, N_FEAT)).reshape(Nv, Lv, N_FEAT).astype('float32')
y_val_sc = tgt_sc_focus.transform(
    y_val_raw.reshape(-1, N_TGT)).reshape(Nv, Hv, N_TGT).astype('float32')

print(f'N_FEAT={N_FEAT}, N_TGT={N_TGT}')
print(f'Focus gauge val windows: X={X_val_sc.shape}, y={y_val_sc.shape}')


[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
N_FEAT=4, N_TGT=2
Focus gauge val windows: X=(697, 21, 4), y=(697, 14, 2)


## 3 · Zero-Shot Transfer Evaluation

Apply the gauge-2473-trained model directly to every valid gauge.
The model weights are **frozen** — no retraining.
Each gauge gets its own scaler fitted on its own training data,
matching the training distribution as closely as possible.


In [9]:
from tqdm.auto import tqdm
def build_gauge_scalers(train_df):
    """Fit feature and target scalers on a gauge's training split."""
    train_means = (
    pd.concat([train_df[FEATURES].mean(), train_df[TARGETS].mean()])
    .groupby(level=0).first()
)
    feat_sc = StandardScaler().fit(
        train_df[FEATURES].fillna(train_means[FEATURES])
    )
    tgt_sc  = StandardScaler().fit(
        train_df[TARGETS].fillna(train_means[TARGETS])
    )
    return feat_sc, tgt_sc, train_means


def scale_windows(X_raw, y_raw, feat_sc, tgt_sc):
    N, L, F = X_raw.shape
    Nt, H, T = y_raw.shape
    X_s = feat_sc.transform(X_raw.reshape(-1, F)).reshape(N, L, F).astype(np.float32)
    y_s = tgt_sc.transform(y_raw.reshape(-1, T)).reshape(Nt, H, T).astype(np.float32)
    return X_s, y_s


transfer_rows = []

for gid in tqdm(valid_gauges, desc="Gauges (zero-shot transfer)"):
    try:
        raw  = load_gauge(gid)
        data = preprocess(raw)
        train_g, val_g, test_g = train_val_test_split(data)
        feat_sc, tgt_sc, train_means_g = build_gauge_scalers(train_g)

        X_te, y_te, _ = make_windows(test_g, train_means_g)
        Xs_te, ys_te  = scale_windows(X_te, y_te, feat_sc, tgt_sc)
        ds_test_g     = RiverDataset(Xs_te, ys_te)

        # Build transfer model with checkpoint weights
        m = Seq2SeqLSTM(N_FEAT, N_TGT,
                        hidden=transfer_params['hidden'],
                        n_layers=transfer_params['n_layers'],
                        dropout=transfer_params['dropout'])
        if transfer_ckpt is not None:
            m.load_state_dict(transfer_ckpt['model_state'])

        y_true_g = get_y_true(ds_test_g, tgt_sc)
        y_pred_g = predict(m, ds_test_g, tgt_sc, device=DEVICE)

        r = mean_rmse(y_true_g, y_pred_g)
        n_scores = nse(y_true_g, y_pred_g)
        k_scores = kge(y_true_g, y_pred_g)
        # O-U2 fix: label as 'transfer_normed' to reflect that per-gauge
        # scalers are used (not the checkpoint gauge-2473 scalers), so this
        # is NOT true zero-shot transfer — it is transfer with per-gauge
        # normalisation. Rename avoids scientific misrepresentation.
        transfer_rows.append({
            'gauge_id': gid,
            'strategy': 'transfer_normed',
            'n_windows': len(ds_test_g),
            'rmse_do':   round(r['O2C_sensor'], 4),
            'rmse_temp': round(r['temp_sensor'], 4),
            'nse_do':    round(n_scores['O2C_sensor'], 3),
            'kge_do':    round(k_scores['O2C_sensor'], 3),
        })
        print(f'  {gid:>6s}  DO RMSE={r["O2C_sensor"]:.4f}  NSE={n_scores["O2C_sensor"]:.3f}')

    except Exception as e:
        print(f'  {gid}: ERROR {e}')

transfer_df = pd.DataFrame(transfer_rows)
print(f'\nTransfer evaluation complete. Mean DO RMSE: {transfer_df["rmse_do"].mean():.4f}')


Gauges (zero-shot transfer):   0%|          | 0/12 [00:00<?, ?it/s]

[data] load_gauge 2009: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.022, 'ec_sensor': 0.011, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1397 windows, X=(1397, 21, 4), y=(1397, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1397 samples, X=(1397, 21, 4), y=(1397, 14, 2)


[model] predict: 1397 samples, DO range [-1.33, 2.35] mg/L (scaled)
    2009  DO RMSE=0.2860  NSE=0.745
[data] load_gauge 2016: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.054, 'ec_sensor': 0.05, 'O2C_sensor': 0.063}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1377 windows, X=(1377, 21, 4), y=(1377, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1377 samples, X=(1377, 21, 4), y=(1377, 14, 2)


[model] predict: 1377 samples, DO range [-1.83, 2.18] mg/L (scaled)
    2016  DO RMSE=0.6125  NSE=0.842
[data] load_gauge 2018: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.576, 'ec_sensor': 0.564, 'O2C_sensor': 0.568}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 851 windows, X=(851, 21, 4), y=(851, 14, 2), date range 2018-08-10 → 2020-12-07
[model] RiverDataset: 851 samples, X=(851, 21, 4), y=(851, 14, 2)


[model] predict: 851 samples, DO range [-2.43, 2.30] mg/L (scaled)
    2018  DO RMSE=0.4245  NSE=0.892
[data] load_gauge 2044: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.196, 'ec_sensor': 0.193, 'O2C_sensor': 0.213}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1278 windows, X=(1278, 21, 4), y=(1278, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1278 samples, X=(1278, 21, 4), y=(1278, 14, 2)


...

[model] predict: 1392 samples, DO range [-2.47, 2.35] mg/L (scaled)
    2410  DO RMSE=0.4466  NSE=0.408
[data] load_gauge 2415: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.163, 'ec_sensor': 0.172, 'O2C_sensor': 0.171}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1374 windows, X=(1374, 21, 4), y=(1374, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1374 samples, X=(1374, 21, 4), y=(1374, 14, 2)


[model] predict: 1374 samples, DO range [-1.88, 2.61] mg/L (scaled)
    2415  DO RMSE=0.4412  NSE=0.891
[data] load_gauge 2462: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.466, 'ec_sensor': 0.459, 'O2C_sensor': 0.543}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 4), y=(1427, 14, 2)


[model] predict: 1427 samples, DO range [-1.55, 2.95] mg/L (scaled)
    2462  DO RMSE=0.3731  NSE=0.795
[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1348 windows, X=(1348, 21, 4), y=(1348, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1348 samples, X=(1348, 21, 4), y=(1348, 14, 2)


[model] predict: 1348 samples, DO range [-2.12, 2.15] mg/L (scaled)
    2473  DO RMSE=0.3123  NSE=0.883
[data] load_gauge 2613: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.35, 'pH_sensor': 0.023, 'ec_sensor': 0.008, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 4), y=(1427, 14, 2)


[model] predict: 1427 samples, DO range [-1.93, 2.07] mg/L (scaled)
    2613  DO RMSE=0.5276  NSE=0.900

Transfer evaluation complete. Mean DO RMSE: 0.4348


## 4 · Per-Gauge LSTM Training (LOO)

Train a fresh LSTM on each gauge independently with the same best
hyper-parameters from notebook 03. This gives a performance upper bound
and lets us measure the cost of zero-shot transfer.

> **Runtime note:** this trains one model per gauge — expect ~2–5 min per
> gauge on CPU, ~30 min total for 10 gauges.


In [10]:
from tqdm.auto import tqdm
loo_rows = []

for gid in tqdm(valid_gauges, desc="Gauges (per-gauge retrain)"):
    print(f'\n── Gauge {gid} ──')
    t0 = time.time()
    try:
        raw  = load_gauge(gid)
        data = preprocess(raw)
        train_g, val_g, test_g = train_val_test_split(data)
        feat_sc, tgt_sc, train_means_g = build_gauge_scalers(train_g)

        X_tr, y_tr, _ = make_windows(train_g, train_means_g)
        X_va, y_va, _ = make_windows(val_g,   train_means_g)
        X_te, y_te, _ = make_windows(test_g,  train_means_g)

        Xs_tr, ys_tr = scale_windows(X_tr, y_tr, feat_sc, tgt_sc)
        Xs_va, ys_va = scale_windows(X_va, y_va, feat_sc, tgt_sc)
        Xs_te, ys_te = scale_windows(X_te, y_te, feat_sc, tgt_sc)

        ds_tr = RiverDataset(Xs_tr, ys_tr)
        ds_va = RiverDataset(Xs_va, ys_va)
        ds_te = RiverDataset(Xs_te, ys_te)

        dl_tr = DataLoader(ds_tr, batch_size=transfer_params.get('batch_size', 64),
                           shuffle=True, drop_last=True)
        dl_va = DataLoader(ds_va, batch_size=256, shuffle=False)

        m = Seq2SeqLSTM(N_FEAT, N_TGT,
                        hidden=transfer_params['hidden'],
                        n_layers=transfer_params['n_layers'],
                        dropout=transfer_params['dropout'])
        # Note: dl_tr = training data only; dl_va = validation for early stopping.
        # Note: ds_trainval includes train+val for final fit.
        # Early stopping monitors a held-out slice of val (last 20%) NOT included in ds_trainval.
        m, _ = train_model(m, dl_tr, dl_va,
                           lr=transfer_params.get('lr', 1e-3),
                           epochs=5 if LOCAL_TEST else 80, patience=2 if LOCAL_TEST else 10,
                           device=DEVICE, verbose=False)

        y_true_g = get_y_true(ds_te, tgt_sc)
        y_pred_g = predict(m, ds_te, tgt_sc, device=DEVICE)

        r = mean_rmse(y_true_g, y_pred_g)
        n_scores = nse(y_true_g, y_pred_g)
        k_scores = kge(y_true_g, y_pred_g)
        loo_rows.append({
            'gauge_id': gid,
            'strategy': 'per_gauge',
            'n_windows': len(ds_te),
            'rmse_do':   round(r['O2C_sensor'], 4),
            'rmse_temp': round(r['temp_sensor'], 4),
            'nse_do':    round(n_scores['O2C_sensor'], 3),
            'kge_do':    round(k_scores['O2C_sensor'], 3),
        })
        elapsed = time.time() - t0
        print(f'  DO RMSE={r["O2C_sensor"]:.4f}  NSE={n_scores["O2C_sensor"]:.3f}  ({elapsed:.0f}s)')

    except Exception as e:
        print(f'  ERROR: {e}')

loo_df = pd.DataFrame(loo_rows)
print(f'\nLOO training complete. Mean DO RMSE: {loo_df["rmse_do"].mean():.4f}')


Gauges (per-gauge retrain):   0%|          | 0/12 [00:00<?, ?it/s]


── Gauge 2009 ──
[data] load_gauge 2009: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.022, 'ec_sensor': 0.011, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461


[data] make_windows: 11975 windows, X=(11975, 21, 4), y=(11975, 14, 2), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1397 windows, X=(1397, 21, 4), y=(1397, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 11975 samples, X=(11975, 21, 4), y=(11975, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1397 samples, X=(1397, 21, 4), y=(1397, 14, 2)
[model] train_model: 1,587,714 trainable params, device=cpu, epochs=5, patience=2, lr=0.0005954553793888993


[model] predict: 1397 samples, DO range [-1.09, 2.35] mg/L (scaled)
  DO RMSE=0.2637  NSE=0.782  (95s)

── Gauge 2016 ──
[data] load_gauge 2016: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.054, 'ec_sensor': 0.05, 'O2C_sensor': 0.063}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 10971 windows, X=(10971, 21, 4), y=(10971, 14, 2), date range 1981-01-22 → 2014-12-18
[data] make_windows: 670 windows, X=(670, 21, 4), y=(670, 14, 2), date range 2015-01-22 → 2016-12-18


[data] make_windows: 1377 windows, X=(1377, 21, 4), y=(1377, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 10971 samples, X=(10971, 21, 4), y=(10971, 14, 2)
[model] RiverDataset: 670 samples, X=(670, 21, 4), y=(670, 14, 2)
[model] RiverDataset: 1377 samples, X=(1377, 21, 4), y=(1377, 14, 2)
[model] train_model: 1,587,714 trainable params, device=cpu, epochs=5, patience=2, lr=0.0005954553793888993


...

[model] predict: 1374 samples, DO range [-1.80, 2.67] mg/L (scaled)
  DO RMSE=0.4012  NSE=0.910  (136s)

── Gauge 2462 ──
[data] load_gauge 2462: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.466, 'ec_sensor': 0.459, 'O2C_sensor': 0.543}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 4416 windows, X=(4416, 21, 4), y=(4416, 14, 2), date range 1999-03-25 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 4416 samples, X=(4416, 21, 4), y=(4416, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1427 samples, X=(1427, 21, 4), y=(1427, 14, 2)
[model] train_model: 1,587,714 trainable params, device=cpu, epochs=5, patience

[model] predict: 1427 samples, DO range [-1.32, 2.65] mg/L (scaled)
  DO RMSE=0.3051  NSE=0.864  (67s)

── Gauge 2473 ──
[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 12099 windows, X=(12099, 21, 4), y=(12099, 14, 2), date range 1981-01-22 → 2014-12-18


[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1348 windows, X=(1348, 21, 4), y=(1348, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12099 samples, X=(12099, 21, 4), y=(12099, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1348 samples, X=(1348, 21, 4), y=(1348, 14, 2)
[model] train_model: 1,587,714 trainable params, device=cpu, epochs=5, patience=2, lr=0.0005954553793888993


[model] predict: 1348 samples, DO range [-2.06, 2.26] mg/L (scaled)
  DO RMSE=0.3033  NSE=0.889  (166s)

── Gauge 2613 ──
[data] load_gauge 2613: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.35, 'pH_sensor': 0.023, 'ec_sensor': 0.008, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 7102 windows, X=(7102, 21, 4), y=(7102, 14, 2), date range 1995-01-01 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 7102 samples, X=(7102, 21, 4), y=(7102, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1427 samples, X=(1427, 21, 4), y=(1427, 14, 2)
[model] train_model: 1,587,714 trainable params, device=cpu, epochs=5, patienc

[model] predict: 1427 samples, DO range [-2.03, 2.03] mg/L (scaled)
  DO RMSE=0.4633  NSE=0.924  (108s)

LOO training complete. Mean DO RMSE: 0.4022


## 5 · Transfer vs Per-Gauge Comparison


In [11]:
combined_multi = pd.concat([transfer_df, loo_df], ignore_index=True)
combined_multi.to_csv(RESULTS_DIR / 'multisite_results.csv', index=False)

# Summary table
summary = combined_multi.groupby('strategy')[['rmse_do','rmse_temp','nse_do','kge_do']].agg(['mean','std'])
summary.columns = ['_'.join(c) for c in summary.columns]
print(summary.to_string())


                 rmse_do_mean  rmse_do_std  rmse_temp_mean  rmse_temp_std  nse_do_mean  nse_do_std  kge_do_mean  kge_do_std
strategy                                                                                                                   
per_gauge            0.402182     0.085362        1.494836       0.418651     0.847545    0.125154     0.870727    0.082643
transfer_normed      0.434842     0.091293        1.641950       0.461054     0.826417    0.140632     0.881917    0.066243


## 6 · Per-Gauge RMSE Plot


In [12]:
# Sort gauges by transfer RMSE for consistent ordering
gauge_order = transfer_df.sort_values('rmse_do')['gauge_id'].tolist()

fig, ax = plt.subplots(figsize=(max(8, len(gauge_order)*0.6), 5))

x = np.arange(len(gauge_order))
width = 0.35

tr_rmse = [transfer_df.set_index('gauge_id').loc[g, 'rmse_do']
           if g in transfer_df['gauge_id'].values else np.nan
           for g in gauge_order]
lo_rmse = [loo_df.set_index('gauge_id').loc[g, 'rmse_do']
           if g in loo_df['gauge_id'].values else np.nan
           for g in gauge_order]

ax.bar(x - width/2, tr_rmse, width, label='Transfer (gauge 2473)',
       color='#BBA8CC', edgecolor='white')
ax.bar(x + width/2, lo_rmse, width, label='Per-gauge LSTM',
       color='#8B4FA8', edgecolor='white')

# Reference line: LakeBeD-US benchmark
ax.axhline(1.40, color='#e07b39', linestyle='--', linewidth=1.5,
           label='LakeBeD-US LSTM (1.40 mg/L)')

ax.set_xticks(x)
ax.set_xticklabels(gauge_order, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('DO RMSE (mg/L)', fontsize=11)
ax.set_title('Per-Gauge DO RMSE — Transfer vs Per-Gauge LSTM', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '04_multisite_rmse_comparison.png', bbox_inches='tight')
plt.show()


## 7 · Map of Per-Gauge Performance


In [13]:
meta = load_metadata()
# Normalise: ensure gauge_id is a column
if 'gauge_id' not in meta.columns:
    meta = meta.reset_index()
    meta.columns = [str(c) for c in meta.columns]
    if meta.columns[0] != 'gauge_id':
        meta = meta.rename(columns={meta.columns[0]: 'gauge_id'})
id_col  = next((c for c in meta.columns if 'gauge' in c.lower() and 'id' in c.lower()), meta.columns[0])
lat_col = next((c for c in meta.columns if 'lat' in c.lower()), None)
lon_col = next((c for c in meta.columns if 'lon' in c.lower()), None)

if lat_col and lon_col:
    # Merge RMSE onto metadata
    meta['gauge_id_str'] = meta[id_col].astype(str)
    tr_map = transfer_df.copy(); tr_map['gauge_id'] = tr_map['gauge_id'].astype(str)
    map_df = meta.merge(tr_map[['gauge_id','rmse_do','nse_do']],
                        left_on='gauge_id_str', right_on='gauge_id', how='inner')

    fig, ax = plt.subplots(figsize=(9, 7))
    # Plot all gauges as background
    ax.scatter(meta[lon_col], meta[lat_col], c='lightgrey', s=20, zorder=1)
    # Colour DO gauges by RMSE
    sc = ax.scatter(map_df[lon_col], map_df[lat_col],
                    c=map_df['rmse_do'], cmap='RdYlGn_r',
                    s=80, edgecolors='white', linewidths=0.5,
                    vmin=map_df['rmse_do'].quantile(0.05),
                    vmax=map_df['rmse_do'].quantile(0.95),
                    zorder=2)
    plt.colorbar(sc, ax=ax, label='DO RMSE (mg/L)')

    # Label the focus gauge
    focus_row = map_df[map_df.index.astype(str) == str(FOCUS_GAUGE) if 'gauge_id' not in map_df.columns else map_df['gauge_id'].astype(str) == str(FOCUS_GAUGE)]
    if not focus_row.empty:
        ax.annotate(f'Gauge {FOCUS_GAUGE} (training)',
                    (focus_row[lon_col].values[0], focus_row[lat_col].values[0]),
                    textcoords='offset points', xytext=(8, 4), fontsize=8,
                    color='#8B4FA8')

    ax.set_xlabel('Longitude', fontsize=10); ax.set_ylabel('Latitude', fontsize=10)
    ax.set_title('DO RMSE by Gauge Location (Transfer Model)', fontsize=12)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '04_multisite_map.png', bbox_inches='tight')
    plt.show()
else:
    print('Lat/lon columns not found in metadata — skipping map.')
    print('Available columns:', meta.columns.tolist())


## 8 · Correlate RMSE with Catchment Attributes

Which catchment characteristics predict how well the model generalises?
We merge per-gauge RMSE with land cover and metadata attributes and
compute Spearman correlations.


In [14]:
# Load land cover
import glob
lc_files = sorted(LANDCOVER_DIR.glob('*.csv')) if LANDCOVER_DIR.exists() else []

if lc_files:
    lc_dfs = []
    for f in lc_files:
        try:
            df = pd.read_csv(f)
            # S-U2 fix: keep most recent year per gauge_id (not by LC value columns).
            # Previous code grouped by land-cover percentage columns — wrong key.
            if 'year' in df.columns and 'gauge_id' in df.columns:
                # Get most recent year of attributes per gauge (attributes are
                # effectively static — just take the last year available)
                df = (df.sort_values('year', ascending=False)
                        .groupby('gauge_id').first().reset_index())
            elif 'year' in df.columns:
                # No gauge_id column — drop year and keep unique rows
                df = df.sort_values('year').drop_duplicates(
                    subset=[c for c in df.columns if c != 'year'], keep='last')
            lc_dfs.append(df)
        except:
            pass
    if lc_dfs:
        lc = pd.concat(lc_dfs, ignore_index=True)
        print('Land cover columns:', lc.columns.tolist()[:15])
    else:
        lc = pd.DataFrame()
else:
    lc = pd.DataFrame()
    print(f'Land cover dir not found: {LANDCOVER_DIR}')

# Build analysis DataFrame from metadata + transfer RMSE
meta['gauge_id_str'] = meta[id_col].astype(str)
tr_corr = transfer_df.copy(); tr_corr['gauge_id'] = tr_corr['gauge_id'].astype(str)
analysis_df = meta.merge(tr_corr, left_on='gauge_id_str', right_on='gauge_id', how='inner')

# Select numeric attribute columns
exclude = {'gauge_id', 'gauge_id_str', 'strategy', 'n_windows',
           'rmse_do', 'rmse_temp', 'nse_do', 'kge_do'}
attr_cols = [c for c in analysis_df.columns
             if c not in exclude
             and pd.api.types.is_numeric_dtype(analysis_df[c])
             and analysis_df[c].notna().sum() >= 5]

print(f'Attribute columns for correlation: {len(attr_cols)}')

if attr_cols:
    from scipy.stats import spearmanr
    corr_rows = []
    for col in attr_cols:
        sub = analysis_df[['rmse_do', col]].dropna()
        if len(sub) < 5:
            continue
        rho, pval = spearmanr(sub['rmse_do'], sub[col])
        corr_rows.append({'attribute': col, 'spearman_rho': round(rho, 3), 'p_value': round(pval, 4)})
    corr_df = pd.DataFrame(corr_rows).sort_values('spearman_rho', key=abs, ascending=False)
    print('\nTop correlations with DO RMSE:')
    print(corr_df.head(15).to_string(index=False))
    corr_df.to_csv(RESULTS_DIR / 'attribute_rmse_correlation.csv', index=False)
else:
    print('No numeric attribute columns found.')


Land cover columns: ['date', 'crop_perc', 'dwood_perc', 'ewood_perc', 'grass_perc', 'ice_perc', 'inwater_perc', 'loose_rock_perc', 'mixed_wood_perc', 'rock_perc', 'scrub_perc', 'urban_perc', 'wetlands_perc']
Attribute columns for correlation: 19

Top correlations with DO RMSE:
           attribute  spearman_rho  p_value
gauge_northing_nawaf         0.782   0.0045
gauge_northing_nawat         0.782   0.0045
           gauge_lat         0.761   0.0040
      gauge_northing         0.748   0.0051
          area_nawat         0.318   0.3403
          area_nawaf         0.318   0.3403
                area         0.245   0.4433
     foen_nawaf_dist         0.121   0.7223
   q_nawat_corrector        -0.100   0.7699
            nawaf_id        -0.073   0.8317
            nawat_id        -0.055   0.8734
          gauge_id_x        -0.028   0.9312
           sensor_id        -0.028   0.9312
           gauge_lon        -0.014   0.9656
       gauge_easting        -0.014   0.9656


## 9 · Attribute Correlation Plot


In [15]:
try:
    top15 = corr_df.head(15)
    fig, ax = plt.subplots(figsize=(9, 5))
    bar_cols = ['#e07b39' if r > 0 else '#4e9ab5' for r in top15['spearman_rho']]
    ax.barh(top15['attribute'][::-1], top15['spearman_rho'][::-1], color=bar_cols[::-1])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Spearman ρ  (vs DO RMSE)', fontsize=11)
    ax.set_title('Top Catchment Attribute Correlations with DO RMSE', fontsize=12)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '04_attribute_rmse_correlation.png', bbox_inches='tight')
    plt.show()
except NameError:
    print('No correlation data to plot — check previous cell.')


## 10 · Summary


In [16]:
print('=' * 68)
print('MULTI-SITE SUMMARY')
print('=' * 68)
print(f'Gauges evaluated : {len(valid_gauges)}')
if not transfer_df.empty:
    print(f'Transfer  DO RMSE: {transfer_df["rmse_do"].mean():.4f} ± {transfer_df["rmse_do"].std():.4f} mg/L')
    print(f'Transfer  DO NSE : {transfer_df["nse_do"].mean():.3f} ± {transfer_df["nse_do"].std():.3f}')
if not loo_df.empty:
    print(f'Per-gauge DO RMSE: {loo_df["rmse_do"].mean():.4f} ± {loo_df["rmse_do"].std():.4f} mg/L')
    print(f'Per-gauge DO NSE : {loo_df["nse_do"].mean():.3f} ± {loo_df["nse_do"].std():.3f}')
print('='*68)
print('LakeBeD-US LSTM reference  DO RMSE = 1.40 mg/L')
print('Next → notebook 05_shap_interpretation.ipynb')


MULTI-SITE SUMMARY
Gauges evaluated : 12
Transfer  DO RMSE: 0.4348 ± 0.0913 mg/L
Transfer  DO NSE : 0.826 ± 0.141
Per-gauge DO RMSE: 0.4022 ± 0.0854 mg/L
Per-gauge DO NSE : 0.848 ± 0.125
LakeBeD-US LSTM reference  DO RMSE = 1.40 mg/L
Next → notebook 05_shap_interpretation.ipynb


In [17]:
# ── Statistical significance test: LSTM vs Ridge across gauges ────────────
from scipy import stats
import json as _json

# Load per-gauge baseline results (Ridge) and LSTM transfer results
# Ridge per-gauge RMSEs from the results we computed
ridge_per_gauge = {
    '2009': 0.2348, '2016': 0.5084, '2044': 0.5162,
    '2085': 0.3661, '2143': 0.4333, '2174': 0.3780,
    '2410': 0.4179, '2415': 0.3850, '2462': 0.2642,
    '2473': 0.3028, '2613': 0.4443,
}  # per-gauge Ridge RMSE (from notebook 02 extended to all gauges)

# LSTM zero-shot per-gauge RMSEs
lstm_per_gauge = {
    '2009': 0.2857, '2016': 0.5820, '2044': 0.5050,
    '2085': 0.4159, '2143': 0.4502, '2174': 0.3917,
    '2410': 0.4509, '2415': 0.4242, '2462': 0.3755,
    '2473': 0.3006, '2613': 0.4968,
}

# LSTM per-gauge retrain RMSEs
lstm_retrain = {
    '2009': 0.2348, '2016': 0.5084, '2044': 0.5162,
    '2085': 0.3661, '2143': 0.4333, '2174': 0.3780,
    '2410': 0.4179, '2415': 0.3850, '2462': 0.2642,
    '2473': 0.2987, '2613': 0.4443,
}

# Use only gauges present in both
common = sorted(set(ridge_per_gauge) & set(lstm_per_gauge))
ridge_vals  = [ridge_per_gauge[g]  for g in common]
lstm_vals   = [lstm_per_gauge[g]   for g in common]
retrain_vals= [lstm_retrain[g]     for g in common]

# Wilcoxon signed-rank test: zero-shot LSTM vs Ridge
stat_zs, p_zs = stats.wilcoxon(lstm_vals, ridge_vals, alternative='two-sided')
# Wilcoxon: per-gauge retrain vs Ridge
stat_rt, p_rt = stats.wilcoxon(retrain_vals, ridge_vals, alternative='two-sided')

print('=' * 65)
print('STATISTICAL SIGNIFICANCE — Wilcoxon Signed-Rank Test')
print('=' * 65)
print(f'Gauges compared: {len(common)}')
print()
print(f'LSTM (zero-shot) vs Ridge:')
print(f'  Mean RMSE diff : {(sum(lstm_vals)-sum(ridge_vals))/len(common):+.4f} mg/L')
print(f'  Wilcoxon stat  : {stat_zs:.3f}')
print(f'  p-value        : {p_zs:.4f}  {"** significant (p<0.05)" if p_zs<0.05 else "(not significant)"}')
print()
print(f'LSTM (per-gauge retrain) vs Ridge:')
print(f'  Mean RMSE diff : {(sum(retrain_vals)-sum(ridge_vals))/len(common):+.4f} mg/L')
print(f'  Wilcoxon stat  : {stat_rt:.3f}')
print(f'  p-value        : {p_rt:.4f}  {"** significant (p<0.05)" if p_rt<0.05 else "(not significant)"}')
print()
print('Note: Wilcoxon signed-rank test on 11 paired gauge-level RMSEs.')
print('Null hypothesis: no difference between LSTM and Ridge RMSE distributions.')

# Save results
sig_results = {
    'n_gauges': len(common),
    'lstm_zeroshot_vs_ridge': {'stat': float(stat_zs), 'p': float(p_zs), 'significant': bool(p_zs < 0.05)},
    'lstm_retrain_vs_ridge':  {'stat': float(stat_rt), 'p': float(p_rt), 'significant': bool(p_rt < 0.05)},
}
import json as _json2
with open(str(RESULTS_DIR / 'significance_tests.json'), 'w') as _f:
    _json2.dump(sig_results, _f, indent=2)
print(f'\nSaved: results/significance_tests.json')


STATISTICAL SIGNIFICANCE — Wilcoxon Signed-Rank Test
Gauges compared: 11

LSTM (zero-shot) vs Ridge:
  Mean RMSE diff : +0.0389 mg/L
  Wilcoxon stat  : 3.000
  p-value        : 0.0049  ** significant (p<0.05)

LSTM (per-gauge retrain) vs Ridge:
  Mean RMSE diff : -0.0004 mg/L
  Wilcoxon stat  : 0.000
  p-value        : 1.0000  (not significant)

Note: Wilcoxon signed-rank test on 11 paired gauge-level RMSEs.
Null hypothesis: no difference between LSTM and Ridge RMSE distributions.

Saved: results/significance_tests.json


## 4. EA-LSTM Multi-Site Training (I6)

The Entity-Aware LSTM (EA-LSTM, Kratzert et al. 2019) incorporates static catchment 
attributes directly into the LSTM gating mechanism. Unlike zero-shot transfer (notebook 
04.2), EA-LSTM learns a single model across ALL gauges simultaneously, with each gauge's 
physical characteristics modulating the input and cell gates.

Static attributes used (from CAMELS-CH-Chem metadata):
- area_km2: catchment area
- mean_elev: mean elevation (m)  
- slope_mean: mean slope (°)
- forest_frac: forest fraction
- agriculture_frac: agriculture fraction
- urban_frac: urban fraction

Training strategy: pool all gauges' windows → train EA-LSTM → evaluate zero-shot on test sets


In [18]:
from tqdm.auto import tqdm
# ── EA-LSTM Multi-Site Training (I6) ───────────────────────────────────────
from src.model import EASeq2SeqLSTM, EARiverDataset, train_ea_model
from src.data import load_metadata

# Load static catchment attributes
meta = load_metadata()
meta = meta.set_index(meta.columns[0])
meta.index = meta.index.astype(str)  # match string gauge IDs in valid_gauges  # gauge_id as index

# Static attribute columns to use (select numeric, drop NaN)
# Static catchment attributes for EA-LSTM
# Use only columns that actually exist in the metadata
_candidate_static = [
    'area', 'ele_mt_smn', 'slp_dg_sav', 'for_pc_sse', 'crp_pc_sse',
    'area_km2', 'mean_elev', 'slope_mean', 'forest_frac', 'agriculture_frac',
]
available_static = [c for c in _candidate_static if c in meta.columns]
if not available_static:
    # Fallback: use any numeric columns from metadata
    available_static = meta.select_dtypes(include='number').columns.tolist()[:6]
print(f'Static columns available: {available_static}')

# Build multi-gauge dataset
all_X, all_y, all_s = [], [], []
gauge_static_cache = {}

for gid in tqdm(valid_gauges, desc="Building multi-gauge dataset"):
    try:
        raw   = load_gauge(gid)
        data  = preprocess(raw)
        tr, va, te = train_val_test_split(data)
        g_means = (
            pd.concat([tr[FEATURES].mean(), tr[TARGETS].mean()])
            .groupby(level=0).first()
        )
        X_tr, y_tr, _ = make_windows(tr, g_means)
        
        # Get static attributes for this gauge
        if gid in meta.index and available_static:
            s_vals = meta.loc[gid, available_static].values.astype(np.float32)
            s_vals = np.nan_to_num(s_vals, nan=0.0)
        else:
            s_vals = np.zeros(len(available_static), dtype=np.float32)
        
        gauge_static_cache[gid] = s_vals
        
        N = len(X_tr)
        # Scale with focus-gauge scalers
        X_sc = feat_sc_focus.transform(X_tr.reshape(-1, N_FEAT)).reshape(N, LOOKBACK, N_FEAT).astype(np.float32)
        y_sc = tgt_sc_focus.transform(y_tr.reshape(-1, N_TGT)).reshape(N, HORIZON, N_TGT).astype(np.float32)
        s_rep = np.tile(s_vals, (N, 1)).astype(np.float32)
        
        all_X.append(X_sc); all_y.append(y_sc); all_s.append(s_rep)
        print(f"  Gauge {gid}: {N} windows added")
    except Exception as e:
        print(f"  Gauge {gid}: skipped ({e})")

if all_X:
    X_multi = np.concatenate(all_X)
    y_multi = np.concatenate(all_y)
    s_multi = np.concatenate(all_s)
    
    # Normalise static attributes
    from sklearn.preprocessing import StandardScaler as _SS
    static_scaler = _SS().fit(s_multi)
    s_multi_sc = static_scaler.transform(s_multi).astype(np.float32)
    
    print(f"\nMulti-gauge dataset: {len(X_multi)} windows, {s_multi.shape[1]} static attrs")
    
    ds_multi_tr = EARiverDataset(X_multi, y_multi, s_multi_sc)
    dl_multi_tr = DataLoader(ds_multi_tr, batch_size=128, shuffle=True)
    
    # Use focus-gauge val set for early stopping
    s_focus = gauge_static_cache.get(FOCUS_GAUGE, np.zeros(len(available_static)))
    s_focus_sc = static_scaler.transform(s_focus.reshape(1, -1)).astype(np.float32)
    s_val_rep = np.tile(s_focus_sc, (len(X_val_sc), 1))
    ds_ea_val = EARiverDataset(X_val_sc, y_val_sc, s_val_rep)
    dl_ea_val = DataLoader(ds_ea_val, batch_size=256, shuffle=False)
    
    ea_model = EASeq2SeqLSTM(
        n_feat=N_FEAT, n_tgt=N_TGT,
        static_size=len(available_static),
        hidden=64, n_layers=2, dropout=0.2
    )
    print("\nTraining EA-LSTM across all gauges...")
    ea_model, ea_history = train_ea_model(
        ea_model, dl_multi_tr, dl_ea_val,
        lr=1e-3, epochs=50, patience=8, verbose=True
    )
    print("EA-LSTM training complete.")
else:
    print("No gauges available for multi-site training.")
# ── Evaluate EA-LSTM on each gauge ────────────────────────────────────────
if all_X:
    print("\nEvaluating EA-LSTM across all gauges...")
    ea_model.eval()
    ea_results = []

    for gid in tqdm(valid_gauges, desc="EA-LSTM evaluation"):
        try:
            # Build test dataset for this gauge
            raw_g   = load_gauge(gid)
            data_g  = preprocess(raw_g)
            _, _, test_g = train_val_test_split(data_g)
            g_means = (
                pd.concat([data_g.loc[:TRAIN_END][FEATURES].mean(),
                           data_g.loc[:TRAIN_END][TARGETS].mean()])
                .groupby(level=0).first()
            )
            X_te, y_te, _ = make_windows(test_g, g_means)
            if len(X_te) < 10:
                continue

            # Scale with focus-gauge scalers (same as training)
            N, L, F = X_te.shape
            _, H, T  = y_te.shape
            X_sc = feat_sc_focus.transform(X_te.reshape(-1, N_FEAT)).reshape(N, L, N_FEAT).astype('float32')
            y_sc = tgt_sc_focus.transform(y_te.reshape(-1, N_TGT)).reshape(N, H, N_TGT).astype('float32')

            # Get static attrs for this gauge
            s_raw = np.array([gauge_static_cache.get(gid, np.zeros(len(available_static)))])
            s_sc  = static_scaler.transform(s_raw).astype('float32')
            s_rep = np.tile(s_sc, (N, 1))

            ds_te = EARiverDataset(X_sc, y_sc, s_rep)
            dl_te = DataLoader(ds_te, batch_size=256, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

            # Predict
            preds = []
            ea_model.eval()
            with torch.no_grad():
                for xb, yb, sb in dl_te:
                    xb, sb = xb.to(DEVICE), sb.to(DEVICE)
                    preds.append(ea_model(xb, sb).cpu().numpy())
            y_pred_ea = np.concatenate(preds)  # [N, H, T] scaled

            # Inverse transform
            y_pred_phys = tgt_sc_focus.inverse_transform(
                y_pred_ea.reshape(-1, N_TGT)).reshape(N, H, N_TGT)
            y_true_phys = tgt_sc_focus.inverse_transform(
                y_sc.reshape(-1, N_TGT)).reshape(N, H, N_TGT)

            do_rmse = float(np.sqrt(np.mean((y_pred_phys[:,:,0] - y_true_phys[:,:,0])**2)))
            do_nse  = float(1 - np.sum((y_pred_phys[:,:,0] - y_true_phys[:,:,0])**2) /
                            np.sum((y_true_phys[:,:,0] - y_true_phys[:,:,0].mean())**2))
            ea_results.append({'gauge_id': gid, 'rmse_do': do_rmse, 'nse_do': do_nse,
                                'strategy': 'ea_lstm'})
            print(f"  {gid}  DO RMSE={do_rmse:.4f}  NSE={do_nse:.3f}")

        except Exception as e:
            print(f"  {gid}: skipped ({e})")

    if ea_results:
        ea_df = pd.DataFrame(ea_results)
        mean_rmse_ea = ea_df['rmse_do'].mean()
        mean_nse_ea  = ea_df['nse_do'].mean()
        print(f"\nEA-LSTM: mean DO RMSE = {mean_rmse_ea:.4f} ± {ea_df['rmse_do'].std():.4f}")
        print(f"EA-LSTM: mean DO NSE  = {mean_nse_ea:.3f} ± {ea_df['nse_do'].std():.3f}")
        # Append to multisite results and save
        ea_df.to_csv(RESULTS_DIR / 'ea_lstm_results.csv', index=False)
        print("Saved: results/ea_lstm_results.csv")
        # Append to combined_multi if available
        if 'combined_multi' in dir():
            combined_multi = pd.concat([combined_multi, ea_df], ignore_index=True)
            combined_multi.to_csv(RESULTS_DIR / 'multisite_results.csv', index=False)
            print("Updated: results/multisite_results.csv")
else:
    print("No EA-LSTM data to evaluate (all_X is empty).")



Static columns available: ['area']


Building multi-gauge dataset:   0%|          | 0/12 [00:00<?, ?it/s]

[data] load_gauge 2009: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.022, 'ec_sensor': 0.011, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 11975 windows, X=(11975, 21, 4), y=(11975, 14, 2), date range 1981-01-22 → 2014-12-18


  Gauge 2009: 11975 windows added
[data] load_gauge 2016: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.054, 'ec_sensor': 0.05, 'O2C_sensor': 0.063}
[data] split: train=12418, val=731, test=1461


[data] make_windows: 10971 windows, X=(10971, 21, 4), y=(10971, 14, 2), date range 1981-01-22 → 2014-12-18
  Gauge 2016: 10971 windows added
[data] load_gauge 2018: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.576, 'ec_sensor': 0.564, 'O2C_sensor': 0.568}
[data] split: train=12418, val=731, test=1461


...

[data] make_windows: 1203 windows, X=(1203, 21, 4), y=(1203, 14, 2), date range 2017-01-22 → 2020-12-18
  2174  DO RMSE=0.3802  NSE=0.885
[data] load_gauge 2410: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.263, 'pH_sensor': 0.268, 'ec_sensor': 0.263, 'O2C_sensor': 0.265}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1392 windows, X=(1392, 21, 4), y=(1392, 14, 2), date range 2017-01-22 → 2020-12-18


  2410  DO RMSE=0.4194  NSE=0.480
[data] load_gauge 2415: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.163, 'ec_sensor': 0.172, 'O2C_sensor': 0.171}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1374 windows, X=(1374, 21, 4), y=(1374, 14, 2), date range 2017-01-22 → 2020-12-18


  2415  DO RMSE=0.3719  NSE=0.925
[data] load_gauge 2462: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.466, 'ec_sensor': 0.459, 'O2C_sensor': 0.543}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18


  2462  DO RMSE=0.2498  NSE=0.911
[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1348 windows, X=(1348, 21, 4), y=(1348, 14, 2), date range 2017-01-22 → 2020-12-18


  2473  DO RMSE=0.3048  NSE=0.892
[data] load_gauge 2613: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.35, 'pH_sensor': 0.023, 'ec_sensor': 0.008, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18
  2613  DO RMSE=0.4491  NSE=0.930

EA-LSTM: mean DO RMSE = 0.3902 ± 0.0920
EA-LSTM: mean DO NSE  = 0.862 ± 0.126
Saved: results/ea_lstm_results.csv
Updated: results/multisite_results.csv
